# 04 — Concepts cliniques : construction, validation, sélection

**Objectif** : donner au modèle un vocabulaire **compréhensible par un médecin**. DermLIP place images et textes dans le même espace ; on mesure, pour chaque image, sa proximité avec des descriptions de concepts cliniques (ulcération, bords irréguliers, pigmentation…).

**Méthode**
1. Vocabulaire défini dans `configs/concepts.yaml` (≈ 20 concepts, 3 formulations chacun, inspirés de l'ontologie de Derm1M).
2. Score d'un concept = similarité image / concept − similarité image / phrase neutre (on retire ce qui est commun à toutes les photos de peau).
3. **Sélection sans utiliser le diagnostic** (pas de fuite) : on garde les concepts **stables** (les formulations donnent des scores concordants) et **non redondants**.
4. Contrôles de validité : cohérence avec les diagnostics attendus, lien avec les symptômes déclarés, concept « artefact » (encre) surveillé.
5. **E5** : classifieur sur les seuls concepts — le modèle le plus interprétable.

**Prérequis** : notebook 03 exécuté (cache `features_dermlip_vitb16.npy`). DermLIP est rechargé pour encoder les textes (pas de nouveau téléchargement si le cache Hugging Face existe).

## 0. Imports, données et fonctions communes

In [ ]:
from pathlib import Path
import json
import re
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (balanced_accuracy_score, f1_score, recall_score,
                             roc_auc_score, roc_curve)
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
N_FOLDS = 5
np.random.seed(SEED)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CSV_PATH = ROOT / "zr7vgbcyr2-1" / "metadata.csv"
PROCESSED_DIR = ROOT / "data" / "processed"
EXTERNAL_DIR = ROOT / "data" / "external" / "derm1m"
CONFIG_DIR = ROOT / "configs"
FIG_DIR = ROOT / "outputs" / "figures"
REPORT_DIR = ROOT / "outputs" / "reports"
CKPT_DIR = ROOT / "outputs" / "checkpoints"
for d in (PROCESSED_DIR, FIG_DIR, REPORT_DIR, CKPT_DIR):
    d.mkdir(parents=True, exist_ok=True)

# variables cliniques retenues (notebook 01) — le profil patient n'est utilisé que dans l'expérience E8
NUM_COLS = ["age"]
BOOL_COLS = ["itch", "grew", "hurt", "changed", "bleed", "elevation"]
CAT_COLS = ["region"]
FEATURES = NUM_COLS + BOOL_COLS + CAT_COLS
PROFILE_NUM = ["fitspatrick", "diameter_1", "diameter_2"]
PROFILE_BOOL = ["smoke", "drink", "pesticide", "skin_cancer_history", "cancer_history"]
PROFILE_CAT = ["gender"]
PROFILE_COLS = PROFILE_NUM + PROFILE_BOOL + PROFILE_CAT

CANCER_CLASSES = ["BCC", "SCC", "MEL"]
DERMLIP_ID = "hf-hub:redlessone/DermLIP_ViT-B-16"

In [ ]:
splits = pd.read_csv(PROCESSED_DIR / "splits.csv")
meta = pd.read_csv(CSV_PATH, na_values=["UNK"])

df = splits.merge(meta[["img_id"] + FEATURES + PROFILE_COLS], on="img_id", how="left",
                  validate="one_to_one")
TO_FLOAT = {True: 1.0, False: 0.0, "True": 1.0, "False": 0.0}
for c in BOOL_COLS + PROFILE_BOOL:                     # True / False / NaN -> 1.0 / 0.0 / NaN
    df[c] = df[c].map(TO_FLOAT)

dev = df[df["split"] == "train"].reset_index(drop=True)
test = df[df["split"] == "test"].reset_index(drop=True)
y_dev, folds, groups_dev = dev["label"].to_numpy(), dev["fold"].to_numpy(), dev["patient_id"].to_numpy()
y_test, groups_test = test["label"].to_numpy(), test["patient_id"].to_numpy()

print(f"Développement : {len(dev)} images, {dev['patient_id'].nunique()} patients | "
      f"test : {len(test)} images (non utilisé avant l'évaluation finale)")
assert set(folds) == set(range(N_FOLDS))

In [ ]:
def metrics_at(y_true, proba, threshold=0.5):
    y_pred = (proba >= threshold).astype(int)
    return {
        "AUC": roc_auc_score(y_true, proba),
        "Exactitude équilibrée": balanced_accuracy_score(y_true, y_pred),
        "Sensibilité": recall_score(y_true, y_pred, pos_label=1),
        "Spécificité": recall_score(y_true, y_pred, pos_label=0),
        "F1": f1_score(y_true, y_pred),
    }


def take_rows(X, idx):
    return X.iloc[idx] if isinstance(X, pd.DataFrame) else X[idx]


def fit_model(model, X, y, groups=None):
    """Entraîne un modèle ; transmet `groups` aux modèles qui choisissent leurs hyperparamètres par patient."""
    needs_groups = isinstance(model, GridSearchCV) or getattr(model, "uses_groups", False)
    if needs_groups:
        assert groups is not None, "ce modèle a besoin des identifiants patients (groups)"
        return model.fit(X, y, groups=groups)
    return model.fit(X, y)


def cross_validate(make_model, X, y, folds, groups=None, return_models=False):
    """Validation croisée sur les 5 plis groupés par patient : métriques par pli + prédictions hors pli."""
    oof, rows, models = np.zeros(len(y)), [], []
    for k in range(N_FOLDS):
        tr, va = np.flatnonzero(folds != k), np.flatnonzero(folds == k)
        model = make_model()
        fit_model(model, take_rows(X, tr), y[tr], None if groups is None else groups[tr])
        oof[va] = model.predict_proba(take_rows(X, va))[:, 1]
        rows.append({"pli": k, **metrics_at(y[va], oof[va])})
        models.append(model)
    per_fold = pd.DataFrame(rows)
    return (per_fold, oof, models) if return_models else (per_fold, oof)


def summarize(name, per_fold):
    m = per_fold.drop(columns="pli")
    return {"Expérience": name, **{c: f"{m[c].mean():.3f} ± {m[c].std():.3f}" for c in m.columns},
            "AUC moyenne": m["AUC"].mean()}


def grouped_bootstrap(y, p, groups, p_ref=None, n_boot=2000, seed=SEED):
    """AUC et IC 95 % par bootstrap des patients ; avec `p_ref`, différence d'AUC appariée."""
    y, p, groups = np.asarray(y), np.asarray(p), np.asarray(groups)
    uniq, inv = np.unique(groups, return_inverse=True)
    members = [np.flatnonzero(inv == g) for g in range(len(uniq))]
    rng = np.random.default_rng(seed)
    aucs, diffs = [], []
    for _ in range(n_boot):
        idx = np.concatenate([members[g] for g in rng.integers(0, len(uniq), len(uniq))])
        if y[idx].min() == y[idx].max():
            continue
        a = roc_auc_score(y[idx], p[idx])
        aucs.append(a)
        if p_ref is not None:
            diffs.append(a - roc_auc_score(y[idx], np.asarray(p_ref)[idx]))
    out = {"AUC": roc_auc_score(y, p), "IC95 bas": np.percentile(aucs, 2.5), "IC95 haut": np.percentile(aucs, 97.5)}
    if p_ref is not None:
        d = np.asarray(diffs)
        out.update({"ΔAUC": out["AUC"] - roc_auc_score(y, p_ref),
                    "ΔAUC IC95 bas": np.percentile(d, 2.5), "ΔAUC IC95 haut": np.percentile(d, 97.5),
                    "P(Δ ≤ 0)": float((d <= 0).mean())})
    return out


def make_clinical_preprocessor(num=NUM_COLS, bools=BOOL_COLS, cats=CAT_COLS):
    return ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), num),
        ("bool", SimpleImputer(strategy="most_frequent"), bools),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cats),
    ])


C_GRID = [0.003, 0.01, 0.03, 0.1, 0.3, 1.0]


def make_probe(c_grid=C_GRID):
    """Sonde linéaire : standardisation + logistique, C choisi par validation croisée groupée (patients)."""
    pipe = Pipeline([("scale", StandardScaler()),
                     ("clf", LogisticRegression(class_weight="balanced", max_iter=5000))])
    return GridSearchCV(pipe, {"clf__C": c_grid}, cv=GroupKFold(n_splits=3), scoring="roc_auc", n_jobs=-1)


def load_features(name):
    """Charge un cache de caractéristiques et renvoie une fonction frame -> matrice alignée sur frame['img_id']."""
    feats = np.load(PROCESSED_DIR / f"features_{name}.npy")
    ids = pd.read_csv(PROCESSED_DIR / f"features_{name}_ids.csv")["img_id"]
    index = pd.Series(np.arange(len(ids)), index=ids.to_numpy())
    return lambda frame: feats[index.loc[frame["img_id"]].to_numpy()]


results, oof_preds = [], {}

## 1. Vocabulaire de concepts

In [ ]:
import yaml

cfg = yaml.safe_load((CONFIG_DIR / "concepts.yaml").read_text(encoding="utf8"))
concepts = cfg["concepts"]
REF_PROMPTS = cfg["reference_prompts"]
vocab = pd.DataFrame([{
    "concept": c["name"], "libellé": c["label_fr"], "famille": c["group"],
    "attendu dans": ", ".join(c.get("expected") or []) or "—",
    "symptôme lié": c.get("related_symptom") or "—",
    "artefact": bool(c.get("artefact", False)), "formulations": len(c["prompts"]),
} for c in concepts])
assert vocab["concept"].is_unique, "noms de concepts dupliqués dans concepts.yaml"
vocab

In [ ]:
# Facultatif : aperçu des concepts annotés dans Derm1M (aide à aligner le vocabulaire)
concept_csv = EXTERNAL_DIR / "concept.csv"
if concept_csv.exists():
    peek = pd.read_csv(concept_csv, nrows=5)
    print("Colonnes de concept.csv :", list(peek.columns)[:60])
    display(peek.head())
else:
    print("concept.csv de Derm1M absent : le vocabulaire de configs/concepts.yaml est utilisé tel quel.")

## 2. Chargement de DermLIP (encodeur de texte)

In [ ]:
try:
    import open_clip
    import torch
except ImportError as e:
    raise ImportError("Installez open_clip :  pip install open_clip_torch") from e

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED)
model, _, preprocess = open_clip.create_model_and_transforms(DERMLIP_ID)
tokenizer = open_clip.get_tokenizer(DERMLIP_ID)
model = model.to(device).eval()
LOGIT_SCALE = float(model.logit_scale.exp().item())
print("DermLIP chargé sur", device, "| échelle des logits :", round(LOGIT_SCALE, 1))


@torch.no_grad()
def encode_texts(texts):
    emb = model.encode_text(tokenizer(texts).to(device)).float()
    return torch.nn.functional.normalize(emb, dim=-1).cpu().numpy()


def prompt_embedding(prompts):
    """Moyenne des plongements de plusieurs formulations, renormalisée (« ensemble de prompts »)."""
    e = encode_texts(prompts).mean(axis=0)
    return e / np.linalg.norm(e)


@torch.no_grad()
def encode_tensors(x):
    emb = model.encode_image(x.to(device)).float()
    return torch.nn.functional.normalize(emb, dim=-1).cpu().numpy()


def load_tensor(path):
    with Image.open(path) as im:
        return preprocess(im.convert("RGB"))

## 3. Plongements des concepts et scores des images

In [ ]:
take_dermlip = load_features("dermlip_vitb16")
X_all = take_dermlip(df)                       # toutes les images (dev + test) : scores nécessaires au notebook 05
X_dev_img = take_dermlip(dev)

ref_emb = prompt_embedding(REF_PROMPTS)
prompt_embs = {c["name"]: encode_texts(c["prompts"]) for c in concepts}
concept_emb = np.stack([prompt_embedding(c["prompts"]) for c in concepts])
names = [c["name"] for c in concepts]

np.savez(PROCESSED_DIR / "concept_text_embeddings.npz", names=np.array(names),
         labels_fr=np.array([c["label_fr"] for c in concepts]), concept_emb=concept_emb, ref_emb=ref_emb)

scores_all = pd.DataFrame(X_all @ concept_emb.T - (X_all @ ref_emb)[:, None], columns=names)
scores_all.insert(0, "img_id", df["img_id"].to_numpy())
scores_dev = scores_all.set_index("img_id").loc[dev["img_id"]].reset_index(drop=True)
print("Scores calculés :", scores_all.shape)

## 4. Sélection des concepts (sans utiliser le diagnostic)

- **Stabilité** : corrélation moyenne entre les scores des différentes formulations d'un même concept (sur les images de développement). Un concept instable dépend de la façon dont il est formulé : il est écarté (seuil fixé à l'avance : 0,5).
- **Redondance** : si deux concepts stables sont corrélés à plus de 0,9, on garde le plus stable.
- Les concepts **artefact** sont calculés mais jamais utilisés par le modèle.

In [ ]:
STAB_MIN, REDUND_MAX, MIN_CONCEPTS = 0.5, 0.9, 6

stability = {}
for c in concepts:
    sims = X_dev_img @ prompt_embs[c["name"]].T - (X_dev_img @ ref_emb)[:, None]
    corr = np.corrcoef(sims.T)
    stability[c["name"]] = float(corr[np.triu_indices_from(corr, k=1)].mean())

table = vocab[["concept", "libellé", "famille", "artefact"]].copy()
table["stabilité"] = table["concept"].map(stability).round(3)
table["retenu"], table["raison"] = False, ""
table.loc[table["artefact"], "raison"] = "artefact : surveillance uniquement"

candidates = table[~table["artefact"]].sort_values("stabilité", ascending=False)
stable = candidates[candidates["stabilité"] >= STAB_MIN]["concept"].tolist()
if len(stable) < MIN_CONCEPTS:
    warnings.warn(f"Seulement {len(stable)} concepts stables : on garde les {MIN_CONCEPTS} plus stables.")
    stable = candidates["concept"].head(MIN_CONCEPTS).tolist()
table.loc[table["concept"].isin(candidates["concept"]) & ~table["concept"].isin(stable), "raison"] = \
    f"instable (< {STAB_MIN})"

corr_dev = scores_dev[stable].corr()
kept = []
for c in stable:
    twin = next((k for k in kept if abs(corr_dev.loc[c, k]) > REDUND_MAX), None)
    if twin:
        table.loc[table["concept"] == c, "raison"] = f"redondant avec {twin}"
    else:
        kept.append(c)
table.loc[table["concept"].isin(kept), ["retenu", "raison"]] = [True, "stable et non redondant"]
print("Concepts retenus :", len(kept), "/", (~table["artefact"]).sum())
table.sort_values(["retenu", "stabilité"], ascending=False)

## 5. Contrôles de validité

Ces contrôles **décrivent** les concepts ; ils ne servent pas à les sélectionner (le diagnostic n'est pas utilisé pour choisir).

In [ ]:
z_dev = (scores_dev - scores_dev.mean()) / scores_dev.std()
by_diag = z_dev.groupby(dev["diagnostic"].to_numpy()).mean().T
order = ["BCC", "SCC", "MEL", "ACK", "NEV", "SEK"]
by_diag = by_diag[order]

fig, ax = plt.subplots(figsize=(7.5, 0.38 * len(by_diag) + 1.5))
im = ax.imshow(by_diag.to_numpy(), cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(order)), order)
ax.set_yticks(range(len(by_diag)), [f"{n}{' ✓' if n in kept else ''}" for n in by_diag.index])
for i, c in enumerate(concepts):
    for diag in c.get("expected") or []:
        ax.add_patch(plt.Rectangle((order.index(diag) - 0.5, i - 0.5), 1, 1, fill=False, lw=1.5, ec="black"))
plt.colorbar(im, label="score moyen (z)")
ax.set_title("Score moyen de chaque concept par diagnostic\n(cadre noir = attendu ; ✓ = retenu)")
plt.tight_layout()
plt.savefig(FIG_DIR / "04_concepts_par_diagnostic.png", dpi=150)
plt.show()

expected_ok = {}
for c in concepts:
    exp = c.get("expected") or []
    if exp:
        expected_ok[c["name"]] = bool(by_diag.loc[c["name"], exp].mean() > by_diag.loc[c["name"]].drop(exp).mean())
table["cohérent avec l'attendu"] = table["concept"].map(expected_ok)
print("Concepts cohérents avec les diagnostics attendus :", sum(expected_ok.values()), "/", len(expected_ok))

In [ ]:
rows = []
for c in concepts:
    s = c.get("related_symptom")
    if s:
        mask = dev[s].notna().to_numpy()
        rows.append({"concept": c["name"], "symptôme": s,
                     "AUC (score → symptôme)": round(roc_auc_score(dev.loc[mask, s], scores_dev.loc[mask, c["name"]]), 3)})
symptom_check = pd.DataFrame(rows)
table["AUC symptôme"] = table["concept"].map(symptom_check.set_index("concept")["AUC (score → symptôme)"])
symptom_check

*Lecture* : une AUC > 0,5 signifie que le concept est plus élevé quand le patient déclare le symptôme (ex. « ulcération » ↔ saignement). Pour `flat_patch` (tache plane), une AUC **< 0,5** est attendue face à `elevation` : c'est aussi un signe de validité.

In [ ]:
artefacts = table.loc[table["artefact"], "concept"].tolist()
for a in artefacts:
    auc_a = roc_auc_score(y_dev, scores_dev[a])
    print(f"Concept artefact « {a} » : AUC pour prédire le cancer = {auc_a:.3f}")
    if abs(auc_a - 0.5) > 0.1:
        print("  ⚠ Lien marqué avec le diagnostic : risque de raccourci (encre). À vérifier dans le notebook 06.")

## 6. E5 — Classifieur sur les seuls concepts (validation croisée imbriquée)

In [ ]:
X_cpt_dev = scores_dev[kept]
per_fold, oof_preds["E5 Concepts seuls"] = cross_validate(make_probe, X_cpt_dev, y_dev, folds, groups=groups_dev)
results.append(summarize("E5 Concepts seuls", per_fold))
print(per_fold.round(3).to_string(index=False))

final = fit_model(make_probe(), X_cpt_dev, y_dev, groups_dev).best_estimator_
coef = pd.Series(final.named_steps["clf"].coef_[0], index=kept).sort_values()
labels = table.set_index("concept")["libellé"]
plt.figure(figsize=(7, 0.35 * len(coef) + 1.5))
plt.barh([labels[c] for c in coef.index], coef, color=np.where(coef > 0, "#C44E52", "#4C72B0"))
plt.axvline(0, color="black", lw=0.8)
plt.xlabel("coefficient standardisé (> 0 : oriente vers « cancer »)")
plt.title("E5 — poids des concepts")
plt.tight_layout()
plt.savefig(FIG_DIR / "04_poids_concepts.png", dpi=150)
plt.show()

In [ ]:
ref_path = PROCESSED_DIR / "oof_dermlip.csv"
if ref_path.exists():
    ref = pd.read_csv(ref_path).set_index("img_id").loc[dev["img_id"]]
    rows = []
    for other in ["E3 DermLIP sonde linéaire", "E1a Clinique – logistique"]:
        if other in ref:
            r = grouped_bootstrap(y_dev, oof_preds["E5 Concepts seuls"], groups_dev, p_ref=ref[other].to_numpy())
            rows.append({"comparaison": f"E5 vs {other}", **{k: round(v, 3) for k, v in r.items()}})
    display(pd.DataFrame(rows))
pd.DataFrame(results).set_index("Expérience").drop(columns="AUC moyenne")

## 7. Sauvegarde

In [ ]:
scores_all.to_csv(PROCESSED_DIR / "concept_scores.csv", index=False)
prompts = {c["name"]: " | ".join(c["prompts"]) for c in concepts}
table["formulations"] = table["concept"].map(prompts)
table.to_csv(PROCESSED_DIR / "concepts_selected.csv", index=False)
pd.DataFrame(results).set_index("Expérience").drop(columns="AUC moyenne").to_csv(REPORT_DIR / "04_concepts_resultats.csv")
oof = dev[["img_id", "patient_id", "diagnostic", "label", "fold"]].assign(**{"E5 Concepts seuls": oof_preds["E5 Concepts seuls"]})
oof.to_csv(PROCESSED_DIR / "oof_concepts.csv", index=False)
print("Enregistrés : concept_scores.csv, concepts_selected.csv, concept_text_embeddings.npz, oof_concepts.csv")

## Conclusion (à compléter après exécution)

- Nombre de concepts retenus et raisons des exclusions (tableau de la section 4).
- Validité : part des concepts cohérents avec les diagnostics attendus ; liens concept ↔ symptôme.
- **E5** : performance d'un modèle qui ne « voit » l'image qu'à travers des concepts nommés. Si E5 est proche de E3, les concepts résument bien l'image ; s'il est nettement plus bas, ils servent surtout à **expliquer** (notebook 06) plutôt qu'à prédire.
- Artefact : AUC du concept « marque de stylo » — un lien fort avec le cancer confirme le risque de raccourci.

**Étape suivante** : `05_fusion.ipynb`.